### Process vs Thread

A **process** is an independent program in execution.  
Each process has:

- its own memory space
- its own resources (file handles, variables, etc.)
- strong isolation from other processes

A **thread** is a lightweight execution unit inside a process.  
Threads in the same process:

- share the same memory and resources
- can communicate faster
- require synchronization (e.g., locks) to avoid race conditions

### Key Differences

| Aspect | Process | Thread |
|---|---|---|
| Memory | Separate memory | Shared memory (within process) |
| Isolation | High (safe) | Low (can affect each other) |
| Creation cost | Higher | Lower |
| Communication | Slower (IPC) | Faster (shared variables) |
| Failure impact | One process crash usually doesn’t kill others | One bad thread can crash the whole process |

### In Python Context

- `threading` is useful for **I/O-bound concurrency**.
- For **CPU-bound true parallelism**, use `multiprocessing` (separate processes, separate GILs).

### Concurrency
Concurrency is the ability of a system to deal with multiple tasks at once.

It is a way of organizing a program so that multiple tasks can be in progress simultaneously, even on a single CPU core, by switching between them efficiently.

#### Key Idea

- Multiple processes or tasks execute over time
- Execution is interleaved and broken into parts
- The execution timeline shows tasks switching between each other
- Tasks may appear to run simultaneously, but they actually take turns

#### Python Implementation

- `threading.Thread`
- `asyncio`

#### Challenges

Concurrency introduces several tricky problems:

- **Race conditions**: Two tasks modify shared data at the same time, causing unpredictable results.
- **Deadlocks**: Two tasks each wait for a resource held by the other, so neither can proceed.
- **Synchronization overhead**: Locks, semaphores, and other coordination tools add complexity and can hurt performance if overused.

#### Note

In Python, threads do **not** execute in parallel for CPU-bound work with standard Python. However, they can run in parallel for I/O-bound work.

### Parallelism

Parallelism means running multiple pieces of code truly simultaneously, using multiple CPU cores. It is different from concurrency, where tasks take turns running and only appear simultaneous.

Parallelism is useful for CPU-bound work such as:

- heavy computation
- number crunching
- image and video processing

#### Python implementation

- `multiprocessing.Process`
- `concurrent.futures.ProcessPoolExecutor`

### Note

Parallelism is not separate from concurrency. It is a specific kind of concurrency where tasks run at the same time on different CPU cores.

So:

- **All parallelism is concurrency**
- **Not all concurrency is parallelism**

### Where each tool fits

| Tool | Concurrency? | True Parallelism? | Why |
|---|---:|---:|---|
| `threading` | Yes | No for CPU-bound code | The GIL allows only one thread to execute Python bytecode at a time. It is useful for I/O waiting, not CPU-heavy work. |
| `asyncio` | Yes | No | Single thread, single core; it switches between tasks efficiently. |
| `multiprocessing` | Yes | Yes | Each process has its own interpreter and memory, so it bypasses the GIL and can run on multiple CPU cores. |
| `concurrent.futures` | Yes | Depends | It is an abstraction layer over threads or processes. |

### Why `concurrent.futures` is flexible

`concurrent.futures` provides two executors:

- `ThreadPoolExecutor` → concurrency only, limited by the GIL for CPU-bound work
- `ProcessPoolExecutor` → true parallelism, because it uses separate processes

Same API, different behavior depending on the executor used:

- `ThreadPoolExecutor` is **concurrency**, not parallelism, for CPU-heavy code
- `ProcessPoolExecutor` is **parallelism** on multiple cores

So `concurrent.futures` is not inherently “for parallelism.” It is a convenience API that can provide either concurrency or parallelism, depending on the executor selected.

### Threading
Threading is a way to run multiple sequences of instructions ("threads") within the same program, sharing the same memory space.
Think of a single program as one worker. Threading lets that worker split their attention across several tasks — switching back and forth between them — instead of doing one task fully before starting the next.

In [ ]:
import threading
import time

# function to simulate downloading a file
def _download_file(name, delay):
    print(f"start {name}")
    time.sleep(delay)
    print(f"end {name}")

# create and start threads to download files concurrently
threads = []
for i in range(3):
    # create a thread for each file download
    t = threading.Thread(target=_download_file, args=(f"file{i}", 2))
    # add the thread to the list of threads
    threads.append(t)
    # start the thread
    t.start()

# wait for all threads to complete
for t in threads:
    t.join()

### Global Interpreter Lock
Global Interpreter Lock — a mutex (lock) built into CPython and PyPy (the standard Python interpreter) that allows only one thread to execute Python bytecode at a time, even on a multi-core machine.


#### Why it exists
CPython manages memory using reference counting — every object tracks how many references point to it, and gets deleted when that count hits zero. This isn't thread-safe by default: if two threads modify an object's reference count at the exact same instant, you can get corrupted memory, crashes, or memory leaks. The GIL was the simplest solution: just let one thread touch Python objects at a time.

#### Note
GIL is a CPython-specific implementation detail, not a fundamental property of "Python" as a language. Other implementations handle it differently.

#### GIL with multiprocessing
When you use multiprocessing, Python doesn't touch or modify the GIL at all. Instead:

- Each process you create gets its own separate Python interpreter
- Each interpreter has its own GIL
- Each process has its own memory space




In [ ]:
import threading
import time

def brew_chai():
    print(f"{threading.current_thread().name} started brewing...")
    count = 0
    for _ in range(100_000_000):
        count += 1
    print(f"{threading.current_thread().name} finished brewing...")

thread1 =threading.Thread(target=brew_chai, name="Barista-1")
thread2 = threading.Thread(target=brew_chai, name="Barista-2")

start = time.time()
thread1.start()
thread2.start()
thread1.join()
thread2.join()
end = time.time()

print(f"total time taken: {end - start:.2f} seconds")

### What is a Lock?

A Lock is a synchronization primitive — basically a flag that says, “only one thread/process may enter this section of code at a time.”  
Whoever wants to run that code must `acquire()` the lock first, and `release()` it when done.  
If someone else already holds it, you wait.

#### The key distinction

| What GIL protects | What Lock protects |
|---|---|
| CPython’s internal state — reference counts, memory structures, and the interpreter itself from crashing/corrupting | Your program’s logic — multi-step operations on shared variables, files, and data structures |

The GIL exists to keep the interpreter safe. It was never designed to make your code thread-safe. That’s your job, using locks.

GIL protects Python’s internal memory from corruption, but it does not protect your program’s logic from race conditions. Those are two different problems, and locks solve the second one.

In [ ]:
import threading

counter = 0

def increment():
    global counter
    for _ in range(1_000_000):
        counter += 1

threads = [threading.Thread(target=increment) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()

print(counter)  # You'd expect 2,000,000... but it's often LESS
# Run this and you'll usually get something less than 2,000,000 — proof that the GIL alone didn't save you from a race condition.

In [ ]:
# Fix it with a Lock
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(1_000_000):
        with lock:
            counter += 1   # now this is a true atomic operation

threads = [threading.Thread(target=increment) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()

print(counter)  # Reliably 2,000,000

#### And for multiprocessing — even more so
Remember, separate processes each have their own GIL (or none relevant to each other at all). So when processes share memory via Value, Array, or a Manager, there's zero built-in protection — no GIL to even partially help. A Lock there is doing 100% of the safety work, not just patching gaps the GIL left behind.

In [ ]:
from multiprocessing import Process, Value, Lock

def increment(shared_counter, lock):
    for _ in range(100000):
        with lock:              # <-- critical: prevents race condition
            shared_counter.value += 1

if __name__ == "__main__":
    counter = Value('i', 0)     # shared integer in real shared memory
    lock = Lock()

    processes = [Process(target=increment, args=(counter, lock)) for _ in range(4)]
    for p in processes: p.start()
    for p in processes: p.join()

    print("Final:", counter.value)  # correctly 400000

### Multiprocessing?
Multiprocessing means running multiple separate processes at the same time, each with its own Python interpreter, memory space, and (as we discussed) its own GIL. Because they're fully independent, they can run truly in parallel across multiple CPU cores — making it the standard way to speed up CPU-bound work in Python (heavy computation, number crunching, image/video processing, etc.).

#### Approaches

| Approach | Best for |
|---|---|
| `Process` | Full manual control, different functions per process |
| `Pool` | Same function to run many times on different data. |
| `ProcessPoolExecutor` | Modern, clean API, same interface as threading, returns Future objects |

In [ ]:
# Process — manual, low-level control
# You directly create and manage individual processes.

from multiprocessing import Process

def square(n):
    print(n * n)

if __name__ == "__main__":
    processes = [Process(target=square, args=(i,)) for i in range(5)]
    for p in processes: p.start()
    for p in processes: p.join()

# Use this when you want fine-grained control over each process individually 
# (e.g., different functions per process, custom lifecycle management).    

#### Value
Value — sharing a single primitive valueValue creates a piece of data in actual shared memory, accessible by multiple processes. You must specify a type code so Python knows how much memory to allocate.


Key things about Value

- Holds one single value at a time (not a collection).
- Backed by real shared memory (via ctypes), so it's fast.
- Not automatically thread/process-safe for compound operations — shared_num.value += 1 is still a read-modify-write, so you need a Lock if multiple processes update it simultaneously (covered earlier).

In [ ]:
from multiprocessing import Process, Value

def increment(shared_num):
    shared_num.value += 1

if __name__ == "__main__":
    num = Value('i', 0)   # 'i' = integer, starting at 0

    processes = [Process(target=increment, args=(num,)) for _ in range(5)]
    for p in processes: p.start()
    for p in processes: p.join()

    print(num.value)  # 5

### Queue — passing messages between processes
Queue is a FIFO (first-in, first-out) pipe that processes can use to send data to each other safely. Instead of sharing memory directly, processes send messages — generally a cleaner and safer pattern.

In [ ]:
from multiprocessing import Process, Queue

def worker(q, n):
    result = n * n
    q.put(result)   # send result back through the queue

if __name__ == "__main__":
    q = Queue()
    processes = [Process(target=worker, args=(q, i)) for i in range(5)]

    for p in processes: p.start()
    for p in processes: p.join()

    results = []
    while not q.empty():
        results.append(q.get())

    print(results)  # [0, 1, 4, 9, 16] (order may vary)

In [ ]:
from multiprocessing import Process, Queue

def producer(q):
    for i in range(5):
        q.put(f"item-{i}")
    q.put(None)  # sentinel value to signal "done"

def consumer(q):
    while True:
        item = q.get()
        if item is None:
            break
        print("Consumed:", item)

if __name__ == "__main__":
    q = Queue()
    p1 = Process(target=producer, args=(q,))
    p2 = Process(target=consumer, args=(q,))
    p1.start(); p2.start()
    p1.join(); p2.join()